# 1. Listings with a Completed Order

**Pattern:** Funnel analysis \
**Product:** 'Thrifted' Marketplace \
**Level of difficulty:** L2


---

## Business question

Of all listings created, how many went on to be sold (i.e., have at least one order placed against that `listing_id`)?
Return a single count: the number of distinct listings with a completed order.

## Solution heuristic

Think about it as the simplest possible funnel: one step, one conversion point (listing created → order placed).

## Assumptions

`order_id` is a reliable, never-null primary key on the `orders` table.

## Metric definition

**Grain:** listing

We have two metrics available in our data schema: 'order_placed' in **listing_events** as a logged event (a timestamped action) and 
actual transaction records in **orders** (with buyer_id, order_value, etc.).
In a real schema, you'd expect every real order to have both a **listing_events** record ('order_placed') and a matching orders record.


## Setup

This notebook runs real SQL against the mock data using [DuckDB](https://duckdb.org/) + [JupySQL](https://jupysql.ploomber.io/). Just run the cells top to bottom.


In [1]:
# Run once per environment
%pip install duckdb-engine --quiet



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb
%load_ext sql
%sql duckdb:///:memory:


Connecting to 'duckdb:///:memory:'

In [3]:
%sql CREATE TABLE listings AS SELECT * FROM read_csv_auto('../../data/thrifted/listings.csv');
%sql CREATE TABLE orders   AS SELECT * FROM read_csv_auto('../../data/thrifted/orders.csv');


Running query in 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

Count


## Peek at the data

Two tables are loaded: `listings` and `orders`. Have a look before writing your query.


In [4]:
%sql SELECT * FROM listings;

Running query in 'duckdb:///:memory:'

listing_id,seller_id,category,price
101,1,dresses,25.0
102,1,shoes,40.0
103,2,jackets,60.0
104,2,dresses,18.5
105,3,accessories,12.0
106,3,jackets,75.0
107,4,shoes,33.0
108,4,dresses,22.0
109,5,accessories,9.5
110,5,jackets,55.0


In [5]:
%sql SELECT * FROM orders;

Running query in 'duckdb:///:memory:'

order_id,buyer_id,listing_id,order_date,order_value
5001,20,101,2026-01-05,25.0
5002,21,103,2026-01-09,60.0
5003,22,104,2026-01-12,18.5
5004,23,106,2026-02-01,75.0
5005,24,108,2026-02-14,22.0


---

## Your turn

Write your own query in the cell below.


In [ ]:
%%sql
##your query here


---

## Solution


In [ ]:
%%sql
SELECT COUNT(DISTINCT l.listing_id) AS sold_listings
FROM listings l
WHERE EXISTS (
    SELECT 1
    FROM orders o
    WHERE o.listing_id = l.listing_id
);


Running query in 'duckdb:///:memory:'

sold_listings
5


### Why this works

- `EXISTS` is a correlated subquery: for each `listing_id` in `listings`, it checks whether at least
  one matching row exists in `orders` — without caring how many matches there are.
- `COUNT(DISTINCT l.listing_id)` then counts each qualifying listing once, even if (hypothetically) it
  had more than one order.
- No join is used in the outer query, so there's no row-multiplication risk. `EXISTS` only returns true/false per listing.